In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
fraud_geo = pd.read_csv(
    "../data/processed/fraud_with_country.csv",
    parse_dates=["signup_time", "purchase_time"]
)

print(f"Loaded DataFrame Shape: {fraud_geo.shape}")

Loaded DataFrame Shape: (129146, 14)


In [4]:
fraud_geo["time_since_signup"] = (
    fraud_geo["purchase_time"]
    - fraud_geo["signup_time"]
).dt.total_seconds()

In [5]:
fraud_geo["hour_of_day"] = (
    fraud_geo["purchase_time"]
    .dt.hour
)

In [6]:
fraud_geo["day_of_week"] = (
    fraud_geo["purchase_time"]
    .dt.dayofweek
)

In [7]:
fraud_geo["transaction_count"] = (
    fraud_geo.groupby("user_id")
    ["user_id"]
    .transform("count")
)

In [8]:
fraud_geo["device_count"] = (
    fraud_geo.groupby("device_id")
    ["device_id"]
    .transform("count")
)

In [9]:
cat_cols = [
    "source",
    "browser",
    "sex",
    "country"
]

In [10]:
fraud_encoded = pd.get_dummies(
    fraud_geo,
    columns=cat_cols,
    drop_first=True
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "transaction_count",
    "device_count"
]

fraud_encoded[num_cols] = scaler.fit_transform(
    fraud_encoded[num_cols]
)

In [12]:
fraud_encoded.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)

In [13]:
# ==========================================
# IMPLEMENTING SMOTE IMBALANCE HANDLING
# ==========================================
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import collections

# 1. Define Features and Target
# Assume 'fraud_encoded' is your final DataFrame after pd.get_dummies()
X = fraud_encoded.drop(columns=["class", "user_id", "device_id", "signup_time", "purchase_time"])
y = fraud_encoded["class"]

# 2. Strict Stratified Train-Test Split (Ensures NO data leakage into test set!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print(f"Original Training Target Class Distribution: {collections.Counter(y_train)}")

# 3. Apply SMOTE exclusively on the training sets to protect model pipeline integrity
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled Training Target Class Distribution (SMOTE): {collections.Counter(y_train_resampled)}")

# 4. Save your structured, resampled data components for Task 2 modeling steps
np.savez("../data/processed/fraud_train_resampled.npz", X=X_train_resampled, y=y_train_resampled)
X_test.to_csv("../data/processed/fraud_test_features.csv", index=False)
y_test.to_csv("../data/processed/fraud_test_target.csv", index=False)

Original Training Target Class Distribution: Counter({0: 93502, 1: 9814})
Resampled Training Target Class Distribution (SMOTE): Counter({0: 93502, 1: 93502})
